<a href="https://colab.research.google.com/github/vineethmekala/ITA0517-COMPUTER-VISION-/blob/main/EXP28_CV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import cv2
import numpy as np
from google.colab import files
from IPython.display import HTML, display
from base64 import b64encode

# Upload video
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

# Read video
cap = cv2.VideoCapture(video_path)

# Background subtraction
fgbg = cv2.createBackgroundSubtractorMOG2(
    history=500, varThreshold=50, detectShadows=False
)

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Resize output
out_width = 640
out_height = int(height * out_width / width)

# Save output video
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(
    'vehicle_detection.mp4',
    fourcc, fps if fps > 0 else 25,
    (out_width, out_height)
)

frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.resize(frame, (out_width, out_height))

    # Apply background subtraction
    mask = fgbg.apply(frame)

    # Remove noise
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.dilate(mask, kernel, iterations=2)

    # Find vehicle contours
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    vehicle_count = 0

    for contour in contours:
        area = cv2.contourArea(contour)

        if area > 800:
            x, y, w, h = cv2.boundingRect(contour)

            vehicle_count += 1

            cv2.rectangle(
                frame, (x, y), (x+w, y+h),
                (0, 255, 0), 2
            )

            cv2.putText(
                frame, "Vehicle",
                (x, y-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6, (0, 255, 0), 2
            )

    cv2.putText(
        frame, "Vehicles: " + str(vehicle_count),
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1, (0, 0, 255), 2
    )

    out.write(frame)
    frame_count += 1

cap.release()
out.release()

print("Vehicle Detection Completed!")
print("Total Frames Processed:", frame_count)

# Download output video
files.download('vehicle_detection.mp4')


Saving man.jpeg to man.jpeg
Vehicle Detection Completed!
Total Frames Processed: 1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>